In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mahmoudreda55/satellite-image-classification")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'satellite-image-classification' dataset.
Path to dataset files: /kaggle/input/satellite-image-classification


In [ ]:
import torchvision
from torchvision import transforms
from torch.utils.data import random_split,DataLoader

data_path = "/kaggle/input/satellite-image-classification"

transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor()
])

dataset = torchvision.datasets.ImageFolder(data_path,transform=transforms)

train,val = random_split(dataset,[4505,1126])

In [ ]:

train_data = DataLoader(train,batch_size=128,shuffle=True)

val_data = DataLoader(val,batch_size=128,shuffle=False)

In [ ]:
import torch
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

class CNN(torch.nn.Module) :
    def __init__(self,in_channel,out_channel,karnal_size,num_layers) :
        super().__init__()

        self.in_channel = in_channel
        self.out_channel = out_channel
        self.karnal_size = karnal_size
        self.num_layers = num_layers


        self.conv_layers = torch.nn.ModuleList()

        in_channel = self.in_channel

        for i in range(self.num_layers) :

            cov = torch.nn.Conv2d(in_channel,self.out_channel,self.karnal_size,padding=1)

            self.conv_layers.append(cov)

            in_channel = self.out_channel


            self.in_channel = self.out_channel

            if (i + 1) % 3 == 0 :

                self.out_channel = self.out_channel + self.out_channel

        self.dense = torch.nn.Linear(in_channel,4)

    def forward(self,x) :

        pool = torch.nn.MaxPool2d(2)

        for index,layer in enumerate(self.conv_layers) :
            x = layer(x)
            x = torch.relu(x)
            if (index + 1) % 5 == 0 :
                x = pool(x)
        g_pool = torch.nn.AdaptiveAvgPool2d(1)
        x = torch.flatten(g_pool(x),start_dim=1)

        z = self.dense(x)

        return z

def training() :
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = CNN(3,3,3,20).to(device)
    optim = torch.optim.Adam(model.parameters(),lr=3e-4)
    criterion = torch.nn.CrossEntropyLoss()

    for i in range(1) :

        for img,label in train_data :

            x,y = (img.to(device),label.to(device))

            x = model(x)

            loss = criterion(x,y)

            optim.zero_grad()
            loss.backward()
            optim.step()


            with torch.no_grad() :

                for val,lbl in val_data :
                    val,lbl = (val.to(device),lbl.to(device))
                    val = model(val)
                    loss_val = criterion(val,lbl)


                    print("Training :",loss.item())
                    print("Evaluation :",loss_val.item())

    return model

In [ ]:
CNN_model = training()

Training : 1.3835331201553345
Evaluation : 1.3684875965118408
Training : 1.3835331201553345
Evaluation : 1.3684875965118408
Training : 1.3835331201553345
Evaluation : 1.3684875965118408
Training : 1.3835331201553345
Evaluation : 1.3684875965118408
Training : 1.3835331201553345
Evaluation : 1.3684875965118408
Training : 1.3835331201553345
Evaluation : 1.3684875965118408
Training : 1.3835331201553345
Evaluation : 1.3684875965118408
Training : 1.3835331201553345
Evaluation : 1.3684875965118408
Training : 1.3835331201553345
Evaluation : 1.36848783493042
Training : 1.3684875965118408
Evaluation : 1.354320764541626
Training : 1.3684875965118408
Evaluation : 1.354320764541626
Training : 1.3684875965118408
Evaluation : 1.354320764541626
Training : 1.3684875965118408
Evaluation : 1.354320764541626
Training : 1.3684875965118408
Evaluation : 1.354320764541626
Training : 1.3684875965118408
Evaluation : 1.354320764541626
Training : 1.3684875965118408
Evaluation : 1.354320764541626
Training : 1.3684

In [ ]:
def validation() :
    perds = []
    label = []

    for x,y in val_data :
        x = x.to('cuda')
        output = CNN_model(x)
        output = output.argmax(dim=1)
        perds.append(output)
        label.append(y)

    x = torch.cat(perds).cpu()
    y = torch.cat(label).cpu()

    results = {
        "accuracy":accuracy_score(x,y),
        "precision":precision_score(x,y, average='macro'),
        "recall_score":recall_score(x,y, average='macro'),
        "f1_score":f1_score(x,y, average='macro'),
        "confusion_matrix":confusion_matrix(x,y),
        "classification_report":classification_report(x,y),
        "roc_auc_score":roc_auc_score(x,y, average='macro', multi_class='ovr')
}
    return results

In [ ]:
validation()

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


{'accuracy': 1.0,
 'precision': 1.0,
 'recall_score': 1.0,
 'f1_score': 1.0,
 'confusion_matrix': array([[1126]]),
 'classification_report': '              precision    recall  f1-score   support\n\n           0       1.00      1.00      1.00      1126\n\n    accuracy                           1.00      1126\n   macro avg       1.00      1.00      1.00      1126\nweighted avg       1.00      1.00      1.00      1126\n',
 'roc_auc_score': nan}

In [ ]:
torch.save(CNN_model.state_dict(),'cnn_model.pth')